We aim to classify images of various fish species using a basic DNN model. The task involves processing image data as input, applying appropriate preprocessing steps, and developing a model to accurately predict the fish species from the images.

1. **Data Collection and Validation:**   
   The dataset used contains images of 9 different fish species. Dataset includes: gilt head bream, red sea bream, sea bass, red mullet, horse mackerel, black sea sprat, striped red mullet, trout, shrimp image samples. The images are resized to 590 x 445 pixels.   

2. **Exploratory Data Analysis and Visualization:**
   Before preprocessing, the image dimensions and color channels were analyzed, and necessary visualizations were created.

3. **Image Preprocessing:**
   Image data must be preprocessed before feeding it into the DNN.    
   TO DO: Apply the right image preprocessing 

In [70]:
import os
import random
import shutil
import glob
import logging
from tqdm import tqdm
from itertools import compress
from typing import Optional, List, Tuple, Dict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
from PIL import Image
import cv2 as cv
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.utils import shuffle
import tensorflow as tf
from tensorflow.keras.preprocessing.image import load_img, img_to_array, ImageDataGenerator
from tensorflow.keras.saving import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization, Layer, Flatten, LeakyReLU, ReLU, GlobalAveragePooling2D
from tensorflow.keras.callbacks import TensorBoard, ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam ,RMSprop
from tensorflow.keras import saving

pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)
plt.style.use("ggplot")
sns.set_palette(sns.diverging_palette(220, 20))

In [ ]:
print("TensorFlow version:", tf.__version__)
print("GPU is", "available" if tf.config.list_physical_devices('GPU') else "NOT AVAILABLE")

## 1. Data Collection:

### Google Drive paths
This version is configured for Google Colab.  
Update only `USER_DATASET_ROOT` if your dataset folder is located elsewhere in `MyDrive`.

In [ ]:
# --- Google Drive configuration for Colab ---
try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print("Drive mount skipped:", e)

DRIVE_ROOT = "/content/drive/MyDrive"

USER_DATASET_ROOT = os.path.join(DRIVE_ROOT, "NA_Fish_Dataset")

# Local Colab workspace for faster preprocessing / training
WORK_ROOT = "/content/TP6_optimized_work"
PREPROCESSED_SIZE = (128, 128)   # Faster than 150x150 and still consistent with the TP6 spirit
BATCH_SIZE = 64                  # Good default for Colab GPU and small datasets
EPOCHS = 35                      # Faster than 100, with EarlyStopping to stop sooner if needed

def resolve_dataset_root(user_root: str) -> str:
    """
    Resolve the dataset root. Accepts either:
    1) a folder that directly contains the class subfolders, or
    2) a folder that contains a subfolder named 'dataset' with the class subfolders.
    """
    candidate_paths = [
        user_root,
        os.path.join(user_root, "dataset"),
        os.path.join(user_root, "Fish_Dataset", "dataset"),
        os.path.join(user_root, "Fish_Dataset", "Fish_Dataset", "dataset"),
    ]
    existing = [p for p in candidate_paths if os.path.isdir(p)]
    if not existing:
        raise FileNotFoundError(
            "Dataset path not found. Update USER_DATASET_ROOT to your real Google Drive path.\n"
            f"Tried: {candidate_paths}"
        )
    return existing[0]

fish_dataset_directory = resolve_dataset_root(USER_DATASET_ROOT)

# Build local working folders in /content for speed
preprocessed_images = os.path.join(WORK_ROOT, "preprocessed_images")
output_path = preprocessed_images
model_images_output = os.path.join(WORK_ROOT, "model_dataset")
models_dir = os.path.join(WORK_ROOT, "models")

os.makedirs(WORK_ROOT, exist_ok=True)
os.makedirs(models_dir, exist_ok=True)

print("Dataset source (Drive):", fish_dataset_directory)
print("Local work root      :", WORK_ROOT)
print("Preprocessed size    :", PREPROCESSED_SIZE)
print("Batch size           :", BATCH_SIZE)
print("Epochs               :", EPOCHS)

# Optional speed-up on GPU
try:
    from tensorflow.keras import mixed_precision
    mixed_precision.set_global_policy("mixed_float16")
    print("Mixed precision policy:", mixed_precision.global_policy())
except Exception as e:
    print("Mixed precision not enabled:", e)


In [72]:
def find_image_classes(images_path: str) -> List[str]:
    """
    Find class subdirectories in the specified directory.
    Hidden/system folders are ignored.
    """
    if not os.path.isdir(images_path):
        raise FileNotFoundError(f"Directory not found: {images_path}")
    return sorted(
        [
            i for i in os.listdir(images_path)
            if os.path.isdir(os.path.join(images_path, i)) and not i.startswith(".")
        ]
    )

In [ ]:
image_classes = find_image_classes(fish_dataset_directory)
image_classes

In [ ]:
def df_from_image_folders(images_path: str, extension: Optional[str] = None) -> pd.DataFrame:
    """
    Create a DataFrame from image files stored in class subfolders.

    Parameters
    ----------
    images_path : str
        Path to the directory containing class subdirectories.
    extension : str, optional
        If provided, keep only this file extension (for example "png" or "jpg").
        If None, common image extensions are included automatically.

    Returns
    -------
    pd.DataFrame
        A DataFrame with columns: 'path' and 'label'.
    """
    if not os.path.isdir(images_path):
        raise FileNotFoundError(f"Directory not found: {images_path}")

    image_classes = find_image_classes(images_path)
    valid_exts = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}
    if extension is not None:
        valid_exts = {"." + extension.lower().lstrip(".")}

    images = []
    labels = []

    for fish in image_classes:
        folder = os.path.join(images_path, fish)
        for f in sorted(os.listdir(folder)):
            full_path = os.path.join(folder, f)
            if os.path.isfile(full_path) and os.path.splitext(f)[1].lower() in valid_exts:
                images.append(full_path)
                labels.append(fish)

    df = pd.DataFrame({"path": images, "label": labels})
    return df

In [ ]:
df = df_from_image_folders(fish_dataset_directory)
df.head()

In [76]:
def display_fish_from_each_class(df: pd.DataFrame, img_size: Tuple[int, int] = (224, 224)) -> None:
    """
    Displays one image from each unique class in the DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing image paths and labels.
    img_size : Tuple[int, int]
        Size to which images will be resized for display.
    """
    
    plt.figure(figsize=(12, 12))
    
    for i, unique_label in enumerate(df["label"].unique()):
        
        plt.subplot(3, 3, i + 1)
        image_path = df[df["label"] == unique_label].iloc[0, 0]
        img = load_img(image_path, target_size=img_size)
        plt.imshow(img)
        plt.title(unique_label)
        plt.axis('off')

    plt.show()

In [ ]:
display_fish_from_each_class(df)

In [78]:
def display_images_from_class(df: pd.DataFrame, class_name: str, num_images: int, img_size: Tuple[int, int] = (224, 224)) -> None:
    """
    Displays a specified number of images from a given class in the DataFrame.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame containing image paths and labels.
    class_name : str
        The class label to filter the images.
    img_size : Tuple[int, int]
        Size to which images will be resized for display.
    num_images : int
        Number of images to display from the given class.
    """
    images = df[df["label"] == class_name]["path"].iloc[:num_images]
    plt.figure(figsize=(12, 12))

    for i, image_path in enumerate(images):
        plt.subplot(4, 4, i + 1)
        plt.imshow(load_img(image_path, target_size=img_size))
        plt.title(image_path[-5:-4]) 
        plt.axis('off')

    plt.show()

In [ ]:
display_images_from_class(df, "Sea Bass", 8)

In [ ]:
df["label"].nunique()

In [ ]:
df[["label"]].value_counts()

## 2. Exploratory Data Analysis and Visualization:

### 2.1. Analysis of Image Sizes and Channels:

- The purpose of this study is to standardize images if they are different sizes. Using the created function below, it has been determined that all images have the same standard size.

In [24]:
def compute_image_statistics_from_df(df: pd.DataFrame) -> pd.DataFrame:
    """
    Computes average width, height, and channel count for images listed in the DataFrame.

    Parameters:
        df (pd.DataFrame): DataFrame containing 'path' and 'label' columns for images.

    Returns:
        pd.DataFrame: DataFrame containing average statistics for each fish class.
    """
    stats = []

    grouped = df.groupby('label')

    for label, group in grouped:
        widths = []
        heights = []
        channel_counts = []

        for _, row in group.iterrows():
            image_path = row['path']
            try:
                image = load_img(image_path)
                image_array = img_to_array(image)
                
                width, height = image.size
                widths.append(width)
                heights.append(height)
                channel_counts.append(image_array.shape[2])
            except Exception as e:
                print(f"Error loading image {image_path}: {e}")
        
        if widths:  
            avg_width = np.mean(widths)
            avg_height = np.mean(heights)
            avg_channels = np.mean(channel_counts)
            min_width = np.min(widths)
            max_width = np.max(widths)
            min_height = np.min(height)
            max_height = np.max(height)

            stats.append({
                'Fish Class': label,
                'Average Width': avg_width,
                'Average Height': avg_height,
                'Average Channels': avg_channels,
                'Min Width': min_width,
                'Max Width' : max_width,
                'Min Height' : min_height,
                'Max Height' : max_height  
            })

    return pd.DataFrame(stats)


In [ ]:
df_statistics = compute_image_statistics_from_df(df)
df_statistics

### 2.2. Visualization of the image rgb channels

In [26]:
def display_rgb_channels(image_path: str) -> None:
    """
    Displays the individual RGB channels of an image.

    Parameters
    ----------
    image_path : str
        The file path to the image.

    Returns
    -------
    None
    """
    image = Image.open(image_path)
    
    r, g, b = image.split()
    
    r_array = np.array(r)
    g_array = np.array(g)
    b_array = np.array(b)

    fig, axes = plt.subplots(1, 3, figsize=(12,12))

    axes[0].imshow(r_array, cmap="Reds")
    axes[0].set_title("Red Channel")
    axes[0].axis("off")

    axes[1].imshow(g_array, cmap="Greens")
    axes[1].set_title("Green Channel")
    axes[1].axis("off")

    axes[2].imshow(b_array, cmap="Blues")
    axes[2].set_title("Blue Channel")
    axes[2].axis("off")


    plt.show()

In [ ]:
sample_image = df["path"].iloc[0]
display_rgb_channels(sample_image)

### 2.3. Histogram of the each fish species images pixel distribution:

- As can be seen from the pixel distribution histogram graphs, each species has different color distributions

In [28]:
def plot_average_pixel_distribution_from_df(df: pd.DataFrame, target_size: tuple = (150, 150)) -> None:
    """
    Plots average pixel value distribution for each class based on a DataFrame containing image paths and labels.

    Parameters:
    df (pd.DataFrame): DataFrame containing 'path' and 'label' columns.
    target_size (tuple): Size to which each image will be resized (default is (150, 150)).

    Returns:
    None: Displays the average pixel value histograms for each class.
    """
    
    unique_labels = df['label'].unique()
    plt.figure(figsize=(15, 10))
 
    for i, label in enumerate(unique_labels):
       
        class_images = df[df['label'] == label]['path'].values
        pixel_values = []
      
        for image_path in class_images:
            image = load_img(image_path, target_size=target_size)
            image_array = img_to_array(image) 
            pixel_values.append(image_array)
        
        pixel_values = np.array(pixel_values)
        avg_pixel_values = np.mean(pixel_values, axis=(0, 1, 2))  
       
        plt.subplot(3, 3, i + 1) 
        plt.hist(avg_pixel_values, bins=50, range=(0, 255), color='blue', alpha=0.7)
        plt.title(f'{label} Pixel Value Distribution')
        plt.xlabel('Pixel Value (0-255)')
        plt.ylabel('Frequency')
    
    plt.tight_layout()
    plt.show()

In [ ]:
plot_average_pixel_distribution_from_df(df)

## 3. Image Preprocessing:

In [32]:
def load_image(image_path: str) -> Image.Image:
    """
    Load an image in RGB format from the given path.

    Parameters
    ----------
    image_path : str
        Path to the image file.

    Returns
    -------
    Image.Image
        Loaded image.
    """
    image = Image.open(image_path).convert("RGB")
    return image

def load_images_from_df(df: pd.DataFrame) -> List[Image.Image]:
    """
    Load images from a DataFrame containing image paths.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with a column "path" for image file paths.

    Returns
    -------
    List[Image.Image]
        List of loaded images.
    """
    images = [load_image(image_path) for image_path in tqdm(df["path"].values.tolist(), total=len(df))]
    return images

In [ ]:
loaded_images = load_images_from_df(df)

In [34]:
def plot_image(image: Image.Image) -> None:
    """
    Display an image using matplotlib with the axis turned off.
    
    Parameters
    ----------
    image : Image.Image
        The image to be displayed.
    
    Returns
    -------
    None
    """
    plt.imshow(image)
    plt.axis("off")
    plt.show()

In [ ]:
plot_image(loaded_images[20])

In [ ]:
def preprocess(pil_image: Image.Image, square: bool = False, target_size: Tuple[int, int] = PREPROCESSED_SIZE) -> np.ndarray:
    """
    Preprocess a PIL image for the DNN model.

    Steps
    -----
    1. Convert to RGB.
    2. Optionally crop to a centered square.
    3. Resize to the target size.
    4. Convert to numpy float32 array.

    Returns
    -------
    np.ndarray
        Preprocessed RGB image array with shape (H, W, 3).
    """
    image = pil_image.convert("RGB")

    if square:
        width, height = image.size
        min_side = min(width, height)
        left = (width - min_side) // 2
        top = (height - min_side) // 2
        right = left + min_side
        bottom = top + min_side
        image = image.crop((left, top, right, bottom))

    image = image.resize(target_size)
    image_array = np.array(image, dtype=np.float32)
    return image_array


In [ ]:
preprocessed_image = preprocess(loaded_images[20])
plot_image(preprocessed_image)

**Applying preprocessing to The All Images and Saving Them to Disk**

In [ ]:
def process_and_save_images(df: pd.DataFrame, output_base_dir: str, target_size: Tuple[int, int] = PREPROCESSED_SIZE) -> None:
    """
    Load, preprocess, and save images organized by class folders.
    Existing output folder is removed first to avoid mixing old and new files.
    Images are saved to the local Colab workspace for faster downstream training.
    """
    if os.path.exists(output_base_dir):
        shutil.rmtree(output_base_dir)
    os.makedirs(output_base_dir, exist_ok=True)

    for _, row in tqdm(df.iterrows(), total=len(df), desc="Preprocessing and saving images"):
        image_path = row["path"]
        label = row["label"]

        image = load_image(image_path)
        processed = preprocess(image, square=False, target_size=target_size)

        label_dir = os.path.join(output_base_dir, label)
        os.makedirs(label_dir, exist_ok=True)

        save_name = os.path.basename(image_path)
        save_path = os.path.join(label_dir, save_name)

        Image.fromarray(processed.astype(np.uint8)).save(save_path)


In [ ]:
# Sanity check for the main working folders used by the notebook
print("fish_dataset_directory:", fish_dataset_directory)
print("preprocessed_images   :", preprocessed_images)
print("model_images_output   :", model_images_output)
print("models_dir            :", models_dir)

In [44]:
# `output_path` is kept as an alias for compatibility with the original TP6 wording.
os.makedirs(preprocessed_images, exist_ok=True)
print("Preprocessed images will be written to:", preprocessed_images)

In [ ]:
process_and_save_images(df, preprocessed_images)

**Control**

- All processed images have been controlled using `df_from_image_folders` function that i created before

In [ ]:
df_preprocessed = df_from_image_folders(preprocessed_images)
df_preprocessed.head()

In [ ]:
df_preprocessed["label"].value_counts()

In [ ]:
display_fish_from_each_class(df_preprocessed)

## 4. Neural Network (NN) Model Creation:

### 4.1. Splitting the Fish_Dataset into Train, Validation, and Test Sets.

10% of the dataset was allocated for validation and another 10% for testing. The validation set was used during model training, while the test set was reserved for evaluating the model's performance after training was completed.

In [49]:
def copy_images(subset_df: pd.DataFrame, target_dir: str) -> None:
    """
    Copy images listed in subset_df['path'] into target_dir.
    """
    os.makedirs(target_dir, exist_ok=True)
    for _, row in tqdm(subset_df.iterrows(), total=len(subset_df), desc=f"Copying images to {target_dir}"):
        shutil.copy2(row["path"], os.path.join(target_dir, os.path.basename(row["path"])))


def split_data_into_train_val_test(
    df_processed: pd.DataFrame,
    output_dir: str,
    train_ratio: float = 0.8,
    val_ratio: float = 0.1,
    test_ratio: float = 0.1,
    random_seed: int = 42,
    clean_output: bool = True
) -> pd.DataFrame:
    """
    Split dataset images into train, validation, and test sets class by class.

    Important
    ---------
    - Existing output_dir is removed first by default to avoid stale folders/files.
    - All classes found in df_processed are recreated in train/validation/test.
    """
    if not np.isclose(train_ratio + val_ratio + test_ratio, 1.0):
        raise ValueError("train_ratio + val_ratio + test_ratio must equal 1.0")

    if clean_output and os.path.exists(output_dir):
        shutil.rmtree(output_dir)

    rng = np.random.default_rng(random_seed)
    class_summary = {"Class": [], "Train Count": [], "Validation Count": [], "Test Count": []}

    labels = sorted(df_processed["label"].unique().tolist())

    # Create empty split/class folders first so that flow_from_directory sees all classes
    for split_name in ["train", "validation", "test"]:
        for class_name in labels:
            os.makedirs(os.path.join(output_dir, split_name, class_name), exist_ok=True)

    for class_name in labels:
        class_group = df_processed[df_processed["label"] == class_name].sample(frac=1, random_state=random_seed).reset_index(drop=True)
        total_images = len(class_group)

        train_count = int(total_images * train_ratio)
        val_count = int(total_images * val_ratio)
        test_count = total_images - train_count - val_count

        class_summary["Class"].append(class_name)
        class_summary["Train Count"].append(train_count)
        class_summary["Validation Count"].append(val_count)
        class_summary["Test Count"].append(test_count)

        train_df = class_group.iloc[:train_count]
        val_df = class_group.iloc[train_count:train_count + val_count]
        test_df = class_group.iloc[train_count + val_count:]

        copy_images(train_df, os.path.join(output_dir, "train", class_name))
        copy_images(val_df, os.path.join(output_dir, "validation", class_name))
        copy_images(test_df, os.path.join(output_dir, "test", class_name))

    summary_df = pd.DataFrame(class_summary).sort_values("Class").reset_index(drop=True)

    # Diagnostics
    for split_name in ["train", "validation", "test"]:
        split_path = os.path.join(output_dir, split_name)
        present_classes = [d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d)) and not d.startswith(".")]
        print(f"{split_name}: {len(present_classes)} classes -> {sorted(present_classes)}")

    return summary_df

In [ ]:
summary_df = split_data_into_train_val_test(df_preprocessed, model_images_output, clean_output=True)
summary_df

### 4.2. Data Augmentation

As previously mentioned, data augmentation was performed on the dataset used in this study. However, it is assumed that these operations were not carried out before.

**Data Augmentation Steps:**

- `Normalization`: Rescaling pixel values from the range of 0-255 to 0-1.
- `Random Rotation`: Rotating images up to 40 degrees randomly.
- `Random Width and Height Shift`: Shifting images randomly by 20% of their width and height.
- `Random Shear`: Applying random shearing transformations with a shear intensity of 0.2.
- `Random Zoom`: Zooming in on images randomly by 20%.
- `Horizontal Flip`: Flipping images horizontally.
- `Fill Method`: Using the 'nearest' fill mode for newly created pixels during transformations.

In [51]:
train_dir = os.path.join(model_images_output, "train")
validation_dir = os.path.join(model_images_output, "validation")

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.10,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=PREPROCESSED_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=True
)

validation_generator = val_datagen.flow_from_directory(
    validation_dir,
    target_size=PREPROCESSED_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    color_mode='rgb',
    shuffle=False
)

class_indices = train_generator.class_indices
class_names = list(class_indices.keys())
num_classes_detected = len(class_indices)
print("Class labels:", class_indices)
print("Detected number of classes:", num_classes_detected)


**Data augmentation Control**

In [ ]:
batch = next(train_generator)
images, labels = batch
num_images = min(5, len(images))

class_names = list(train_generator.class_indices.keys())

plt.figure(figsize=(12, 4))
for i in range(num_images):
    plt.subplot(1, 5, i + 1)
    plt.imshow(images[i])
    class_id = int(np.argmax(labels[i]))
    plt.title(class_names[class_id])
    plt.axis('off')
plt.tight_layout()
plt.show()

### 4.3. Creation of the ANN Architecture

**Professional DNN design choices used in this final improved version:**

1. **Input preprocessing**

   All images are resized to `128x128` and normalized with `rescale=1./255`. This keeps the input pipeline simple and stable for a dense neural network.

2. **DNN architecture adapted to image flattening**

   Since the model uses `Flatten()` instead of convolutional layers, the number of trainable parameters can become very large very quickly. The previous compact version avoided overfitting well, but it also showed **underfitting**.  
   To improve learning capacity while staying faithful to a pure DNN design, the architecture is now:

   - `Flatten`
   - `Dense(128) + BatchNormalization + ReLU + Dropout(0.20)`
   - `Dense(64) + BatchNormalization + ReLU + Dropout(0.10)`
   - `Dense(num_classes, softmax)`

3. **Regularization tuned more carefully**

   The model still uses regularization, but less aggressively than before:

   - **L2 weight regularization** with a lighter value `1e-5`
   - **Moderate dropout** to reduce co-adaptation without blocking learning
   - **Batch normalization** to stabilize optimization and improve convergence
   - **Data augmentation** only on the training set

4. **Training control**

   To obtain a more robust final model:

   - `EarlyStopping` stops training when validation loss no longer improves
   - `ModelCheckpoint` saves the best model
   - `ReduceLROnPlateau` lowers the learning rate when validation performance saturates
   - `Adam(learning_rate=5e-4)` improves stability for this denser ANN

**Why this version is better**

This improved DNN remains faithful to the ANN-only requirement of the TP while using a more appropriate capacity for the observed results.  
It is stronger than the previous compact model, less likely to underfit, and still regularized enough to limit overfitting on a small image dataset.


In [ ]:
@saving.register_keras_serializable()
class CustomDNN(Model):
    def __init__(self, input_shape=(128, 128, 3), num_classes=9, name="custom_ann_pro", **kwargs):
        super(CustomDNN, self).__init__(name=name, **kwargs)
        self.num_classes = num_classes
        self.model_input_shape = input_shape

        # Improved ANN-only architecture:
        # slightly stronger than the previous compact version in order
        # to reduce underfitting while keeping good regularization.
        self.flatten = Flatten()

        self.dense1 = Dense(
            128,
            kernel_regularizer=tf.keras.regularizers.l2(1e-5)
        )
        self.bn1 = BatchNormalization()
        self.act1 = ReLU()
        self.drop1 = Dropout(0.20)

        self.dense2 = Dense(
            64,
            kernel_regularizer=tf.keras.regularizers.l2(1e-5)
        )
        self.bn2 = BatchNormalization()
        self.act2 = ReLU()
        self.drop2 = Dropout(0.10)

        self.classifier = Dense(num_classes, activation="softmax", dtype="float32")

    def call(self, inputs, training=False):
        x = self.flatten(inputs)

        x = self.dense1(x)
        x = self.bn1(x, training=training)
        x = self.act1(x)
        x = self.drop1(x, training=training)

        x = self.dense2(x)
        x = self.bn2(x, training=training)
        x = self.act2(x)
        x = self.drop2(x, training=training)

        outputs = self.classifier(x)
        return outputs

    def get_config(self):
        config = super(CustomDNN, self).get_config()
        config.update({
            "input_shape": self.model_input_shape,
            "num_classes": self.num_classes
        })
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

    def train(self, train_dir, val_dir, batch_size=BATCH_SIZE, epochs=EPOCHS):
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=15,
            width_shift_range=0.10,
            height_shift_range=0.10,
            shear_range=0.08,
            zoom_range=0.15,
            horizontal_flip=True,
            fill_mode="nearest"
        )

        validation_datagen = ImageDataGenerator(rescale=1./255)

        target_size = self.model_input_shape[:2]

        train_generator = train_datagen.flow_from_directory(
            train_dir,
            target_size=target_size,
            batch_size=batch_size,
            class_mode="categorical",
            color_mode="rgb",
            shuffle=True,
        )

        validation_generator = validation_datagen.flow_from_directory(
            val_dir,
            target_size=target_size,
            batch_size=batch_size,
            class_mode="categorical",
            color_mode="rgb",
            shuffle=False,
        )

        detected_num_classes = len(train_generator.class_indices)
        if detected_num_classes != self.num_classes:
            raise ValueError(
                f"Class mismatch: model expects {self.num_classes} classes, "
                f"but the training directory contains {detected_num_classes} classes. "
                f"Classes found: {list(train_generator.class_indices.keys())}"
            )

        os.makedirs(models_dir, exist_ok=True)

        checkpoint_val_loss = ModelCheckpoint(
            filepath=os.path.join(models_dir, "best_model.keras"),
            monitor="val_loss",
            save_best_only=True,
            save_weights_only=False,
            mode="min",
            verbose=1
        )

        early_stopping_callback = EarlyStopping(
            monitor="val_loss",
            patience=6,
            restore_best_weights=True,
            verbose=1
        )

        lr_callback = ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
            verbose=1
        )

        optimizer = Adam(learning_rate=5e-4)

        self.compile(
            optimizer=optimizer,
            loss="categorical_crossentropy",
            metrics=["accuracy"]
        )

        history = self.fit(
            train_generator,
            epochs=epochs,
            validation_data=validation_generator,
            callbacks=[checkpoint_val_loss, early_stopping_callback, lr_callback],
            verbose=1
        )

        return history.history


In [ ]:
import time

# Split directories created in the local Colab workspace
train_dir = os.path.join(model_images_output, "train")
val_dir = os.path.join(model_images_output, "validation")
test_dir = os.path.join(model_images_output, "test")

# Quick check of detected classes in each split
for split_name, split_dir in [("train", train_dir), ("validation", val_dir), ("test", test_dir)]:
    split_classes = find_image_classes(split_dir)
    print(f"{split_name}: {len(split_classes)} classes -> {split_classes}")

# Generators for final evaluation (without augmentation)
eval_datagen = ImageDataGenerator(rescale=1./255)

train_eval_generator = eval_datagen.flow_from_directory(
    train_dir,
    target_size=PREPROCESSED_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

val_eval_generator = eval_datagen.flow_from_directory(
    val_dir,
    target_size=PREPROCESSED_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

test_eval_generator = eval_datagen.flow_from_directory(
    test_dir,
    target_size=PREPROCESSED_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

detected_classes = list(train_eval_generator.class_indices.keys())
num_classes = len(detected_classes)

print("\nDetected class mapping:", train_eval_generator.class_indices)
print("Number of detected classes:", num_classes)

# Build and train the optimized TP6 CustomDNN
model = CustomDNN(input_shape=PREPROCESSED_SIZE + (3,), num_classes=num_classes)

start_time = time.time()
result = model.train(train_dir=train_dir, val_dir=val_dir, batch_size=BATCH_SIZE, epochs=EPOCHS)
execution_time = time.time() - start_time

# Load the best saved model before reporting final metrics
best_model_path = os.path.join(models_dir, "best_model.keras")
if os.path.exists(best_model_path):
    model = load_model(best_model_path)

# Global evaluation
train_loss, train_acc = model.evaluate(train_eval_generator, verbose=0)
val_loss, val_acc = model.evaluate(val_eval_generator, verbose=0)
test_loss, test_acc = model.evaluate(test_eval_generator, verbose=0)

minutes = int(execution_time // 60)
seconds = int(execution_time % 60)

print("\n" + "="*70)
print("GLOBAL RESULTS OF THE OPTIMIZED TP6 CUSTOMDNN MODEL")
print("="*70)
print(f"Execution time        : {minutes} min {seconds} s ({execution_time:.2f} s)")
print(f"Train images          : {train_eval_generator.samples}")
print(f"Validation images     : {val_eval_generator.samples}")
print(f"Test images           : {test_eval_generator.samples}")
print(f"Detected classes      : {num_classes}")
print(f"Class names           : {detected_classes}")
print(f"Image size            : {PREPROCESSED_SIZE}")
print(f"Batch size            : {BATCH_SIZE}")
print(f"Max epochs            : {EPOCHS}")
print("="*70)

summary_df = pd.DataFrame({
    "Dataset": ["Train", "Validation", "Test"],
    "Loss": [train_loss, val_loss, test_loss],
    "Accuracy": [train_acc, val_acc, test_acc]
})

display(summary_df)

history_df = pd.DataFrame({
    "Metric": ["final_train_loss", "final_train_accuracy", "final_val_loss", "final_val_accuracy"],
    "Value": [
        result['loss'][-1] if 'loss' in result else np.nan,
        result['accuracy'][-1] if 'accuracy' in result else np.nan,
        result['val_loss'][-1] if 'val_loss' in result else np.nan,
        result['val_accuracy'][-1] if 'val_accuracy' in result else np.nan
    ]
})

display(history_df)


In [ ]:
# Optional: copy the best trained model back to Google Drive for persistence
backup_model_path = os.path.join(DRIVE_ROOT, "best_model_tp6_optimized.keras")
best_model_path = os.path.join(models_dir, "best_model.keras")

if os.path.exists(best_model_path):
    shutil.copy2(best_model_path, backup_model_path)
    print("Best model copied to Drive:", backup_model_path)
else:
    print("Best model not found yet.")


In [ ]:
model = load_model(os.path.join(models_dir, "best_model.keras"))
model.summary()

### Final DNN Optimization Notes

This final notebook keeps a **pure DNN approach** and improves generalization with three main decisions:

- reducing the hidden layer sizes
- adding `L2` regularization on dense layers
- using moderate dropout instead of stacking too many dense blocks

This makes the model more appropriate for flattened image inputs and helps balance **underfitting** and **overfitting** more effectively.


**Visualize with History**

In [59]:
def plot_training_history(result: Dict[str, list]) -> None:
    """
    Plot training and validation loss/accuracy from a history dictionary.
    """
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    plt.plot(result['loss'], label='Train Loss')
    plt.plot(result['val_loss'], label='Validation Loss')
    plt.title('Loss Graph')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()

    plt.subplot(1, 2, 2)
    plt.plot(result['accuracy'], label='Train Accuracy')
    plt.plot(result['val_accuracy'], label='Validation Accuracy')
    plt.title('Accuracy Graph')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()

    plt.tight_layout()
    plt.show()

In [ ]:
plot_training_history(result)

### 4.4. Model Evaluation

The following tasks have been carried out:

- After training is completed, the model's performance is evaluated on the test set.
  
- The performance of the model is analyzed using metrics such as classification accuracy, confusion matrix, accuracy, precision, and recall.
  
- Errors are examined by visualizing the examples that the model misclassified.

**After the test data is formatted appropriately and predictions are made, the performance metrics have been calculated.**



**Evaluation Metrics**

**Precision:**

- **Definition:** The ratio of true positive predictions to all positive predictions (true positives + false positives).
- **Formula:**
  $$
  \text{Precision} = \frac{TP}{TP + FP}
  $$

**Recall:**

- **Definition:** The ratio of true positives to all positive cases (true positives + false negatives).
- **Formula:**
  $$
  \text{Recall} = \frac{TP}{TP + FN}
  $$

**Accuracy:**

- **Definition:** The ratio of correct predictions to the total predictions.
- **Formula:**
  $$
  \text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN}
  $$


**Confusion Matrix and Classification Report Insights:**

**Overall Metrics:**

`Accuracy:`  - The model correctly classified images across all classes.

`Macro Average Precision:`  - Average precision across all classes, not considering class imbalance.

`Macro Average Recall:`  - Average recall across all classes.

---

**Class-Level Insights:**

- `Class 0 (Black Sea Sprat):` 
  
- `Class 1 (Gilt-Head Bream):` 

- `Class 2 (Horse Mackerel):` 

- `Class 3 (Red Mullet):` 

- `Class 4 (Red Sea Bream):` 

- `Class 5 (Sea Bass):` 

- `Class 6 (Shrimp):` 

- `Class 7 (Striped Red Mullet):` 

- `Class 8 (Trout):` 

---

**Conclusion:**



In [61]:
def evaluate_model(
    model,
    test_dir: str,
    target_size: tuple = PREPROCESSED_SIZE,
    batch_size: int = 32
) -> None:
    """
    Evaluate the model on the test dataset and display performance metrics.
    """
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_directory(
        test_dir,
        target_size=target_size,
        batch_size=batch_size,
        class_mode='categorical',
        shuffle=False
    )

    predictions = model.predict(test_generator)
    predicted_classes = np.argmax(predictions, axis=1)
    true_classes = test_generator.classes
    class_names = list(test_generator.class_indices.keys())

    accuracy = accuracy_score(true_classes, predicted_classes)
    precision = precision_score(true_classes, predicted_classes, average='macro', zero_division=0)
    recall = recall_score(true_classes, predicted_classes, average='macro', zero_division=0)

    print("Accuracy:", accuracy)
    print("Precision (macro):", precision)
    print("Recall (macro):", recall)

    conf_matrix = confusion_matrix(true_classes, predicted_classes)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt="d", cmap="Blues",
                xticklabels=class_names, yticklabels=class_names)
    plt.title("Confusion Matrix")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    print(classification_report(true_classes, predicted_classes, target_names=class_names, zero_division=0))

    return test_generator, predicted_classes


In [ ]:
test_directory = os.path.join(model_images_output, "test")
test_generator, predicted_classes = evaluate_model(model, test_directory, target_size=PREPROCESSED_SIZE)

**Analyze Misclassifications**

In [63]:
def analyze_and_visualize_misclassifications(test_generator, predicted_classes, num_samples=10) -> None:
    """
    Analyze and visualize misclassified images from the test dataset.
    """
    test_generator.reset()

    class_labels = {v: k for k, v in test_generator.class_indices.items()}

    df = pd.DataFrame({
        'filename': test_generator.filenames,
        'predict': predicted_classes,
        'y': test_generator.classes
    })

    misclassified = df[df['y'] != df['predict']]
    total_misclassified = misclassified.shape[0]

    print(f'Total misclassified images: {total_misclassified}')

    if total_misclassified > 0:
        display_df = misclassified.copy()
        display_df["true_label"] = display_df["y"].map(class_labels)
        display_df["predicted_label"] = display_df["predict"].map(class_labels)
        print(display_df[["filename", "true_label", "predicted_label"]])

        samples_to_display = min(num_samples, total_misclassified)
        misclassified_samples = misclassified.sample(samples_to_display, random_state=42)

        cols = 5
        rows = (samples_to_display + cols - 1) // cols
        fig, axes = plt.subplots(rows, cols, figsize=(15, 3 * rows))
        axes = np.array(axes).reshape(-1)

        for ax, (_, row) in zip(axes, misclassified_samples.iterrows()):
            img_path = os.path.join(test_generator.directory, row['filename'])
            img = plt.imread(img_path)
            ax.imshow(img)
            ax.set_title(f'True: {class_labels[row["y"]]}\nPred: {class_labels[row["predict"]]}')
            ax.axis('off')

        for i in range(samples_to_display, len(axes)):
            axes[i].axis('off')

        plt.tight_layout()
        plt.show()


In [ ]:
analyze_and_visualize_misclassifications(test_generator, predicted_classes, num_samples=10)

### 4.5. **Prediction**

A random image for each fish in the test folder has been selected and compiled into a list and it was observed that the model successfully made accurate predictions for fish species.

In [65]:
def predict_single_image(model, image_path, test_generator):
    """
    Predict the class of a single image using a trained model.
    """
    image = load_img(image_path, target_size=PREPROCESSED_SIZE)
    plt.imshow(image)
    plt.title(f"Selected Image: {os.path.basename(image_path)}")
    plt.axis('off')
    plt.show()

    image_array = img_to_array(image) / 255.0
    image_array = np.expand_dims(image_array, axis=0)

    predictions = model.predict(image_array, verbose=0)
    predicted_index = np.argmax(predictions, axis=1)[0]

    class_indices = test_generator.class_indices
    class_labels = {v: k for k, v in class_indices.items()}

    predicted_label = class_labels[predicted_index]
    confidence = predictions[0][predicted_index]

    print(f"Predicted Label: {predicted_label} (Confidence: {confidence:.2f})")


**Prediction list for each class from the test image directory**

In [66]:
test_folder = os.path.join(model_images_output, "test")
test_df = df_from_image_folders(test_folder)
prediction_list = test_df.groupby("label")["path"].sample(1).to_list()

In [ ]:
for image in prediction_list:
    class_name = os.path.basename(os.path.dirname(image))
    predict_single_image(model, image, test_generator)
    print(f"Actual label: {class_name}")
    print(50 * " ")
    print(50 * "*")